In [ ]:
!uv pip install geopandas matplotlib numpy requests tqdm contextily

In [ ]:
from pathlib import Path
from getpass import getpass
from io import BytesIO
from zipfile import ZipFile
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import shapely
from shapely import Polygon
from google.colab import userdata
import contextily as cx
import seaborn as sns

# Beyond the Choropleth!

So far this the semester most of our maps have been variations on the choropleth map. Choropleths are every urban informaticist's workhorse map, but they aren't always the best tool for the job. Today, we'll discuss some of the problems with the choropleth map and demonstrate some approaches for overcoming them. We'll also introduce some alternative mapping techniques for visualizing spatial data.

## Load data

Let's start by loading our data...same exact steps from last week:

In [ ]:
# load tract geometries (shapefile)
gdf_tracts = gpd.read_file(
    "https://raw.githubusercontent.com/gboeing/ppd430/main/data/tl_2017_06_tract/tl_2017_06_tract.shp"
)
gdf_tracts.shape

In [ ]:
# load tract census data (csv file)
df_tracts = pd.read_csv(
    "https://raw.githubusercontent.com/gboeing/ppd430/main/data/census_tracts_data_ca.csv",
    dtype={"GEOID10": str, "state": str, "county": str},
)
df_tracts.shape

In [ ]:
# finally, merge tract geometries and census data together
gdf_tracts_data = pd.merge(
    left=gdf_tracts, right=df_tracts, left_on="GEOID", right_on="GEOID10", how="left"
)
gdf_tracts_data.shape

In [ ]:
# load listings, create geometry column, convert to GeoDataFrame
df_listings = pd.read_csv(
    "https://raw.githubusercontent.com/gboeing/ppd430/main/data/listings-la_oc_vc.csv"
)
df_listings["geometry"] = gpd.points_from_xy(x=df_listings["longitude"], y=df_listings["latitude"])
gdf_listings = gpd.GeoDataFrame(df_listings, crs="epsg:4326")
gdf_listings.shape

In [ ]:
# they don't match, so project one to the other's CRS
gdf_listings = gdf_listings.to_crs(gdf_tracts_data.crs)

In [ ]:
# now spatial join listings to tracts
gdf_all = gdf_tracts_data.sjoin(gdf_listings, how="left", predicate="contains")
gdf_all.shape

Instead of analyzing listing attributes by tract (e.g. median rent/sqft), what if we want to assess the _frequency_ of listings across tracts? This should tell us which areas have the most rental listings.


In [ ]:
# "index_right" column contains the index of the corresponding row from the
# listings df after our left spatial join.
# count() aggregation function treats NULL as 0, and everything else as 1
listings_per_tract = gdf_all.groupby("GEOID")['index_right'].count().reset_index().rename(columns={'index_right':'count_listings'})
listings_per_tract.shape

In [ ]:
# add the listing counts back to the tract data gdf
gdf = gdf_tracts_data.merge(listings_per_tract, on='GEOID')
gdf.shape

In [ ]:
# clean up the tracts data
gdf = gdf[gdf['COUNTYFP'].isin(['059','037'])].copy()  # OC/LA
gdf = gdf.loc[~gdf['GEOID'].isin(['06037599000','06037599100','06111990100'])]  # Channel Islands
gdf = gdf[gdf['ALAND'] > 0].copy()  # water-only tracts
gdf.shape

In [ ]:
# clip the raw listings to OC/LA
gdf_listings = gdf_listings.clip(gdf)
gdf_listings.shape

# Problem \#1: Choosing a color scheme

Choosing a color scheme is one of the hardest parts about making a good choropleth: Continuous (unclassed) or classed? If classed, how many breaks? Quantiles or equal interval? Sequential or divergent color palette?

Your choices can have a profound impact on how your map gets interpreted:

| Classed | Unclassed |
| :---: | :---: |
| ![Caption 1](https://datawrapper.dwcdn.net/HGQkI/full.png) |![image.png](https://datawrapper.dwcdn.net/uDBp2/full.png)|




Let's turn back to the data we've been working with all semester. Which map is the best for visualizing our listings count data?

In [ ]:
fig, axarr = plt.subplots(2,3,figsize=(12,8))
axarr = axarr.flatten()

_ = gdf.plot(column='count_listings', ax=axarr[0], cmap="plasma", legend=True, scheme="NaturalBreaks",k=5)
axarr[0].set_axis_off()
axarr[0].set_title("Natural Breaks")

_ = gdf.plot(column='count_listings', ax=axarr[1], cmap="plasma", legend=True, scheme="EqualInterval",k=3)
axarr[1].set_axis_off()
axarr[1].set_title("Equal Interval (k=3)")

_ = gdf.plot(column='count_listings', ax=axarr[2], cmap="plasma", legend=True, scheme="PrettyBreaks")
axarr[2].set_axis_off()
axarr[2].set_title("PrettyBreaks")

_ = gdf.plot(column='count_listings', ax=axarr[3], cmap="plasma", legend=True, legend_kwds={'shrink':0.5})
axarr[3].set_axis_off()
axarr[3].set_title("Continuous")

_ = gdf.plot(column='count_listings', ax=axarr[4], cmap="plasma", legend=True, scheme="Quantiles")
axarr[4].set_axis_off()
axarr[4].set_title("Quantiles")

_ = gdf.plot(column='count_listings', ax=axarr[5], cmap="plasma", legend=True, scheme="MaximumBreaks")
axarr[5].set_axis_off()
axarr[5].set_title("MaximumBreaks")

fig.tight_layout()
plt.show()

## Solutions

- Use a diverging color scheme if your data has a natural zero point.
- Start with a continuous color gradient. This leaves interpretation to the reader, but it avoids choosing a misleading classification scheme
- Use statistical breaks to simplify the map for your reader. If your classes are meaningful to your analysis, that's a good reason to choose a classed map.

Check out [this excellent blog post](https://www.datawrapper.de/blog/classed-vs-unclassed-color-scales) from the datawrapper for more tips on picking the right color scheme.

# Problem \#2. The Modifiable Areal Unit Problem

One common issue when working with geospatial data is known as the Modifiable Areal Unit Problem (MAUP). For our purposes, it describes the phenomenon by which the same underlying data (e.g. points) can produce very different results depending on the unit of geographic aggregation (e.g. block, tract, county):


 | Ex. 1| Ex. 2 |
 |--|--|
 |![](https://mikejohnson51.github.io/spds/lec-img/15-maup.png)|<img src="https://upload.wikimedia.org/wikipedia/commons/a/ab/Maup_rate_numbers.png" width="700">|

The implication here is that your choice of unit of analysis can also profoundly impact your results...so choose wisely!

Census geographies are often used for analyzing data simply because that is the level at which the underlying data has been collected. That's not a great reason to use them, especially if you don't need the extra context provided by administrative boundaries. Census geographies are rife with MAUP!!!

## Solutions
### Use disaggregate data
Using raw, disaggregate data avoids the problem of MAUP completely, but it's not always practical. Let's try plotting our disaggregate listings data. What issues do you see?

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
gdf_listings.plot(ax=ax)
gdf.plot(lw=0.5, edgecolor="gray", facecolor='none', zorder=0, ax=ax)
ax.set_axis_off()

In [ ]:
# Now it's your turn. Try playing with the markersize, marker, and alpha
# parameters of gdf.plot() to improve our map of disaggregate listings

### More solutions:
- Choose a unit of analysis that is relevant to your analysis
- If administrative boundaries aren't relevant, consider an equal-area grid

# Problem 3. Unequal Areas

Tracts and Block Groups are specifically designed to have similar numbers of people living in them, which is great for comparing populations, but also means that some geographies are MUCH bigger than others. This has some undesirable effects:
   1. Many phenomenon are correlated with land area, not just population, which makes it more likely to collect more observations in larger areas (e.g. new residential construction)
   2. Visual bias on a map: large areas over-emphasized, small areas de-emphasized




### Normalize by Area

Perhaps the simplest solution to handling differently sized areal units is to normalize your data by the area of each unit. This converts a count-based attributes into densities, and converts the unit of measure from count to count-per-unit-area. Let's try that with our listings:

In [ ]:
gdf['listing_density'] = gdf['count_listings'] / gdf['ALAND'] / 3.86102e-7

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
_ = gdf.plot(column='listing_density',ax=ax, cmap="plasma", legend=True,
             legend_kwds={
    "shrink": 0.4,
    "label":"listings/sqft",
    # "orientation":"horizontal",
    # "location":"top"
    }
             )
ax.set_axis_off()
plt.tight_layout()
plt.show()

What do you notice? The map looks a bit closer to what we'd expect, with downtown LA and Hollywood starting to pop. But it's still pretty hard to see those tiny tracts. Also, it's not obvious that a density is the right way to think about how rental listings are distributed across LA and Orange Counties

In [ ]:
# Now it's your turn. Try normalizing by population (listings/person) instead of
# area. Does the map tell a different story?

### Use Equal-Area Bins

An even better idea might be to forget Census geographies altogether and just use equal-area bins. This is usually the recommended approach for visualizing count/frequency data.

Hexagons possess many attributes that make them well suited for this task, which you can read more about [here](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/h-whyhexagons.htm)

![](https://pro.arcgis.com/en/pro-app/3.6/tool-reference/spatial-statistics/GUID-3EF700EB-1EEF-44BB-AB02-EE17F7E03F82-web.png)

Matplotlib make it incredibly easy to implement equal area hexagonal bins. Let's try it with our listings data:

In [ ]:
fig, ax = plt.subplots(
    figsize=(10,10)
    )
hb = ax.hexbin(
    gdf_listings['geometry'].x,
    gdf_listings['geometry'].y,
    mincnt=1,
    gridsize=(30,25),
    alpha=.8,
    lw=0.5,
    cmap='plasma'
)
gdf.plot(fc='none', edgecolor='gray', lw=0.5, zorder=0, ax=ax)
fig.colorbar(hb, shrink=.3, label='listings',ax=ax, anchor=(0.3,3), orientation="horizontal",)
ax.set_axis_off()
fig.tight_layout()

In [ ]:
# Now it's your turn. Experiment with the gridsize, cmap, and mincnt params of
# ax.hexbin() to find the hexbin map you think looks best.

## Interpolate with a 2D KDE

If we want to avoid aggregating the data into somewhat arbitrary geographies like census tracts, we can use 2D kernel density estimation to convert our discrete data points into a continuous surface:

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))
g = sns.kdeplot(
    x=gdf_listings['geometry'].x,
    y=gdf_listings['geometry'].y,
    cmap='plasma',
    alpha=0.8,
    cbar=True,
    cbar_kws={'shrink':0.5,'label':"listings"})
gdf.plot(fc='none',edgecolor='k',lw=0.25,zorder=0,ax=ax)
ax.set_axis_off()


This type of visualization is also called a contour map, because the data is visualized as contour lines and can be read much like a topographic elevation map.

Now it's your turn. Try adjusting the cmap, fill, bw_adjust, levels, and thresh parameters to improve on the map above. In particular, pay attention to the bw_adjust param: too big a number and your map will overgeneralize the actual variation in your data, too small and you're better off using discrete bins. Below is some code to get you started:

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))
g = sns.kdeplot(
    x=gdf_listings['geometry'].x,
    y=gdf_listings['geometry'].y,
    cmap='rocket',
    fill=True,
    alpha=0.8,
    bw_adjust=0.7,
    cbar=True,
    levels=100,
    thresh=0,
    cbar_kws={'shrink':0.5,'label':"listings"})
gdf.plot(
    fc='none',
    edgecolor='white',
    lw=0.25,
    zorder=1,
    alpha=0.6,
    ax=ax)
ax.set_axis_off()


# Problem 4: All-or-nothing Allocation

Whether using equal-area units or administrative boundaries, choropleth maps still require you to assign a single value to each unit. This works well for univariate ([or even bi-variate](https://www.axismaps.com/guide/bivariate-choropleth)) continuous data, but what about categorical data? What if we want to map the racial composition of a region? Census geographies aren't only one race. And if we try to map them that way, we risk exaggerating the degree of segregation that may or may not present.

## Solutions

### The Dot Density Map

[Dot density maps](https://www.axismaps.com/guide/dot-density) are a great alternative to choropleth for showing how variables are distributed across geographic areas. They are still subject to the MAUP, but because they can show individual observations (1 dot ~ 1 observation), they can be more intuitive to understand. And because you can control the ratio of dots to observations, they avoid the problem of overlapping dots you get when mapping disaggregate data.


Professor Bill Rankin (aka [Radical Cartography](http://www.radicalcartography.net/) popularized the use of dot density maps for visualizing racial population distributions with his 2010 map "A Taxonomy of Trasitions":

![](https://socks-studio.com/img/blog/chicagodots_race_700.jpg)

For more on this subject, let's turn to Bill himself: https://www.youtube.com/watch?v=8pRcdMVkA3k

# Building a Racial Dot Density Map


For the remainder of this notebook, we're going to focus on using Python to create our own racial dot density map*. The steps will be as follows:
1. Download Census data
2. Define a dot-generation function
3. Generate dots for our Census Tracts based on Census data
4. Plot our dots on a map
5. Make the map look nice

Let's get started

## Get your own census data!

In [ ]:
CENSUS_API_KEY = userdata.get('census_api_key')

In [ ]:
YEAR = 2024  # Latest ACS year
STATE_FIPS = "06"  # California
COUNTY_FIPS = "037"  # Los Angeles

In [ ]:
# Mutually exclusive categories from ACS table B03002.
api_variables = {
    "White non-Hispanic": ["B03002_003E"],
    "Black non-Hispanic": ["B03002_004E"],
    "Asian non-Hispanic": ["B03002_006E"],
    "Hispanic or Latino": ["B03002_012E"],
    "Other non-Hispanic": [
        "B03002_005E",  # American Indian and Alaska Native alone
        "B03002_007E",  # Native Hawaiian and Other Pacific Islander alone
        "B03002_008E",  # Some other race alone
        "B03002_009E",  # Two or more races
    ],
}
all_variables = [v for group in api_variables.values() for v in group]
print(all_variables)

In [ ]:
url = f"https://api.census.gov/data/{YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME"] + all_variables),
    "for": "tract:*",
    "in": f"state:{STATE_FIPS} county:{COUNTY_FIPS}",
    "key": CENSUS_API_KEY,
}
response = requests.get(url, params=params, timeout=60)
rows = response.json()

Pandas can easily create a df from json

In [ ]:
acs = pd.DataFrame(rows[1:], columns=rows[0])

HTTP requests load everything as text, so our GEOIDS are good but our numeric values are not!

In [ ]:
for column in all_variables:
    acs[column] = pd.to_numeric(acs[column])

If we didn't know the names of our Census variables ahead of time, we could use the Census [website](https://data.census.gov/) to figure out the API endpoint we need, and then plug it in here:

In [ ]:
alt_url = f"<PASTE URL HERE>&key={CENSUS_API_KEY}"

In [ ]:
alt_response = requests.get(alt_url)

In [ ]:
alt_rows = response.json()
alt_acs = pd.DataFrame(alt_rows[1:], columns=rows[0])
alt_acs = alt_acs[[col for col in alt_acs.columns if col in ['NAME','GEO_ID'] + all_variables]]
alt_acs.head()

Clean up the ACS data

In [ ]:
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
acs.rename(columns={
    "B03002_003E":"White non-Hispanic",
    "B03002_004E":"Black non-Hispanic",
    "B03002_006E":"Asian non-Hispanic",
    "B03002_012E":"Hispanic or Latino"
}, inplace=True)
acs["Other non-Hispanic"] = acs[api_variables["Other non-Hispanic"]].sum(axis=1, min_count=1)

race_columns = ["White non-Hispanic", "Black non-Hispanic", "Asian non-Hispanic",
                "Hispanic or Latino", "Other non-Hispanic"]
acs = acs[["GEOID", "NAME", *race_columns]].copy()
acs.head()

## Load tract geoms!

In [ ]:
tracts = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_06_tract_500k.zip")

In [ ]:
tracts = tracts.loc[tracts["COUNTYFP"] == COUNTY_FIPS].copy()
tracts = tracts.to_crs(epsg=3310)

# Join on the 11-digit Census tract GEOID.
tracts = tracts.merge(acs, on="GEOID", how="inner", validate="one_to_one")
print(f"Mapped {len(tracts)} Los Angeles County census tracts")

## Generate random points in polygons

Much of the code in this section was borrowed or adapted from [this](https://github.com/agaidus/census_data_extraction/tree/master) very old Github repo by Github user agaidus.

In [ ]:
p1 = Polygon([[0,2],[1,3],[2,2]])
p2 = Polygon([[0,0],[0,2],[2,2],[2,0]])
p3 = Polygon([[2,0],[2,3],[3,3],[3,0]])
shapes_gdf = gpd.GeoSeries([p1,p2,p3])

vals = pd.Series([100, 50, 500])  # "population"

ax = shapes_gdf.plot(facecolor=["#ec5858", "#22b2da", "#6ecb63"])
for label,loc in zip(vals, shapes_gdf.centroid):
    ax.text(loc.x-.2, loc.y, str(label), fontsize=15)


In [ ]:
def random_points_in_polygon(polygon, n, rng):

    # Return n uniformly sampled Shapely points inside a polygon.
    if n <= 0:
        return []

    min_x, min_y, max_x, max_y = polygon.bounds

    accepted_points = []
    while len(accepted_points) < n:

        remaining = n - len(accepted_points)  # n points we still need
        batch_size = max(100, remaining * 2)  # n points to sample at once

        # create points from two lists of randomly generated x and y coords
        candidates = shapely.points(
            rng.uniform(min_x, max_x, batch_size),
            rng.uniform(min_y, max_y, batch_size),
        )

        # only keep points inside the shape
        inside = candidates[shapely.contains(polygon, candidates)]

        # only keep as many points as we need
        inside_needed = inside[:remaining].tolist()
        accepted_points.extend(inside_needed)

    return accepted_points

In [ ]:
DOT_VALUE = 1  # one dot per 1 person
RANDOM_SEED = 42  # for reproducibility
rng = np.random.default_rng(RANDOM_SEED)  # define outside of function to preserve randomness
points = []
for i, shape in enumerate(shapes_gdf):
  pop = vals[i]
  ndots = int(round(pop / DOT_VALUE))
  new_points = random_points_in_polygon(shape, ndots, rng)
  points.extend(new_points)
dots = gpd.GeoSeries(points)

In [ ]:
ax = shapes_gdf.plot(facecolor=["#ec5858", "#22b2da", "#6ecb63"])
dots.plot(ax=ax, markersize=1, color='k')

## Create points for tracts!

In [ ]:
DOT_VALUE = 1000  # one dot represents 1,000 residents
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
point_geometries = []
point_groups = []

for _, tract in tracts.iterrows():
    polygon = tract.geometry
    for group in race_columns:
        count = tract[group]
        if pd.isna(count) or count <= 0:
            continue
        n_dots = int(np.rint(count / DOT_VALUE))
        if n_dots == 0:
            continue
        new_points = random_points_in_polygon(polygon, n_dots, rng)
        point_geometries.extend(new_points)
        point_groups.extend([group] * len(new_points))

dots = gpd.GeoDataFrame(
    {"group": point_groups},
    geometry=point_geometries,
    crs=tracts.crs,
)

## Make your map!

In [ ]:
colors = {
    "White non-Hispanic": "#EE00A2",  # pink
    "Black non-Hispanic": "#00A3DC",  # blue
    "Asian non-Hispanic": "#8FFF3D",  # lime green
    "Hispanic or Latino": "#FFA32A",  # orange
    "Other non-Hispanic": "#757575",  # gray
}

fig, ax = plt.subplots(figsize=(12,12))

# plot dots by racial category
for group, color in colors.items():
    subset = dots.loc[dots["group"] == group]
    subset.plot(ax=ax, color=color, markersize=1)

# add tract outlines
tracts.plot(facecolor='none', ax=ax, lw=0.25)

plt.show()

## Clean up your map!

In [ ]:
# drop Channel Islands
tracts = tracts.loc[~tracts['GEOID'].isin(['06037599000','06037599100'])]

In [ ]:
# drop water-only tracts
tracts = tracts.loc[tracts['ALAND'] > 0]

In [ ]:
# regenerate points with different dot value!
DOT_VALUE = 10  # one dot represents 25 residents
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
point_geometries = []
point_groups = []

for _, tract in tracts.iterrows():
    polygon = tract.geometry
    for group in race_columns:
        count = tract[group]
        if pd.isna(count) or count <= 0:
            continue
        n_dots = int(np.rint(count / DOT_VALUE))
        if n_dots == 0:
            continue
        new_points = random_points_in_polygon(polygon, n_dots, rng)
        point_geometries.extend(new_points)
        point_groups.extend([group] * len(new_points))

dots = gpd.GeoDataFrame(
    {"group": point_groups},
    geometry=point_geometries,
    crs=tracts.crs,
)

In [ ]:
# add highway features
hwy = gpd.read_file("https://gis.data.ca.gov/api/download/v1/items/1f71fa512e824ff09d4b9c3f48b6d602/geojson?layers=0")
hwy = hwy.to_crs("epsg:4326").to_crs(tracts.crs)
hwy = hwy.loc[hwy['RouteID'].str.contains("SHS")]  # interstate, national highway system, non-interstate strategic routes, or the 110
hwy = hwy[hwy['RouteID'].str.contains('110|010|405|101|210|005|210|710|605|002|134|170|090|091|001')]
tracts_buffered = tracts.copy()
tracts_buffered['geometry'] = tracts_buffered['geometry'].buffer(5000)
hwy_la = hwy.clip(tracts_buffered)

In [ ]:
colors = {
    "White non-Hispanic": "#EE00A2",  # pink
    "Black non-Hispanic": "#00A3DC",  # blue
    "Asian non-Hispanic": "#8FFF3D",  # lime green
    "Hispanic or Latino": "#FFA32A",  # orange
    # "Other non-Hispanic": "#757575",  # gray
}

fig, ax = plt.subplots(figsize=(12,12))

for group, color in colors.items():
    subset = dots.loc[dots["group"] == group]
    subset.plot(
        ax=ax,
        color=color,
        marker='.',
        markersize=0.1,
        alpha=0.7,
        label=group,
        zorder=3,
        linewidths=0)

tracts.plot(facecolor='none', ax=ax, lw=0.25, edgecolor="lightgray", zorder=2)
_ = hwy_la.plot(ax=ax, color='k', lw=0.2, zorder=4)  # highway geometries
_ = tracts.dissolve().plot(ax=ax, fc='white', lw=0.5, zorder=1, alpha=0.5)  # County boundary

ax.set_title(
    f"Los Angeles County population by race and ethnicity\n"
    f"2020–2024 ACS 5-year estimates; ",
    fontsize=14,
)
ax.set_axis_off()
ax.legend(
    title=f"1 dot ≈ {DOT_VALUE:,} resident(s)",
    loc=(0.1, 0.09),
    markerscale=50
    )

plt.show()

In [ ]:
# Now it's your turn. Experiment with different settings (e.g. figsize, marker,
# markersize, alpha, DOT_VALUE). Can you make an map that looks good where
# 1 dot = 1 person? Can you find any interesting transition zones? Can you make a new map zoomed
# in on just that area?

## Adding a basemap with Contextily

Web maps make the transition from one zoom level to another look smooth, but what's happening behind the scenes is not!

| Zoom Level 2 | Zoom Level 3 |
| :---: | :---: |
| ![Caption 1](https://www.axismaps.com/images/guide/zoom2.jpeg) | ![Caption 2](https://www.axismaps.com/images/guide/zoom3.jpeg)|

Contextily is a Python library that lets you add a static set of tiles to your maps

In [ ]:
colors = {
    "White non-Hispanic": "#EE00A2",  # pink
    "Black non-Hispanic": "#00A3DC",  # blue
    "Asian non-Hispanic": "#8FFF3D",  # lime green
    "Hispanic or Latino": "#FFA32A",  # orange
    # "Other non-Hispanic": "#757575",  # gray
}

fig, ax = plt.subplots(figsize=(12,12))

for group, color in colors.items():
    subset = dots.loc[dots["group"] == group]
    subset.plot(
        ax=ax,
        color=color,
        marker='.',
        markersize=0.1,
        alpha=0.7,
        label=group,
        zorder=3,
        linewidths=0)

tracts.plot(facecolor='none', ax=ax, lw=0.25, edgecolor="lightgray", zorder=2)
# _ = hwy_la.plot(ax=ax, color='k', lw=0.2, zorder=4)  # highway geometries
_ = tracts.dissolve().plot(ax=ax, fc='white', lw=0.5, zorder=1, alpha=0.5)  # County boundary

ax.set_title(
    f"Los Angeles County population by race and ethnicity\n"
    f"2020–2024 ACS 5-year estimates; ",
    fontsize=14,
)
ax.set_axis_off()
ax.legend(
    title=f"1 dot ≈ {DOT_VALUE:,} resident(s)",
    loc=(0.1, 0.09),
    markerscale=50
    )

cx.add_basemap(
    ax,
    crs=dots.crs,
    # source=cx.providers.CartoDB.PositronNoLabels(apikey=userdata.get("cartodb")),
    source=cx.providers.CartoDB.PositronOnlyLabels(apikey=userdata.get("cartodb")),
    # source=cx.providers.Esri.WorldGrayCanvas,
    zoom=10,
)
plt.show()

Now it's your turn. Check out all the different tile providers supported by Contextily [here](https://contextily.readthedocs.io/en/latest/providers_deepdive.html#What-is-this-%22provider%22-object-?), and experiment to find which tilesets work best for your visualization.